# PyTorch Fundamentals
## Tensors with Gradients

### Creating Tensor with Gradients
- A Variable wraps a Tensor
- Allows accumulation of gradients
- In PyTorch, `requires_grad` is a flag on tensors that tells Autograd whether to track operations on that tensor so it can compute gradients during backpropagation.

What it does

- `requires_grad=True`: PyTorch will record all operations on this tensor and build a computation graph. When you call loss.backward(), it computes gradients w.r.t. this tensor and stores them in tensor.grad.

- `requires_grad=False` (default for plain tensors): PyTorch won’t track operations on this tensor, and no gradients will be computed for it (saves memory and compute).

- Model parameters created by layers like nn.Linear, nn.Conv2d, etc., default to requires_grad=True, since they need to be trained.


Why it matters

- Training: You want gradients for learnable parameters (weights/biases), so they need requires_grad=True.
- Freezing layers (e.g., transfer learning): Set requires_grad=False on those parameters to stop training them.
- Inference / evaluation: Use torch.no_grad() to temporarily disable grad tracking for speed and memory savings.

In [1]:
import torch

In [2]:
a = torch.ones((2, 2), requires_grad=True)
a

tensor([[1., 1.],
        [1., 1.]], requires_grad=True)

In [3]:
a.requires_grad

True

In [4]:
# Not a variable
no_gradient = torch.ones(2, 2)

In [5]:
no_gradient.requires_grad

False

In [6]:
# Behaves similarly to tensors
b = torch.ones((2, 2), requires_grad=True)
print(a + b)
print(torch.add(a, b))

tensor([[2., 2.],
        [2., 2.]], grad_fn=<AddBackward0>)
tensor([[2., 2.],
        [2., 2.]], grad_fn=<AddBackward0>)


In [7]:
print(a * b)
print(torch.mul(a, b))

tensor([[1., 1.],
        [1., 1.]], grad_fn=<MulBackward0>)
tensor([[1., 1.],
        [1., 1.]], grad_fn=<MulBackward0>)


### Manually and Automatically Calculating Gradients

**What exactly is `requires_grad`?**
- Allows calculation of gradients w.r.t. the variable

$$y_i = 5(x_i+1)^2$$

In [8]:
x = torch.ones(2, requires_grad=True)
x

tensor([1., 1.], requires_grad=True)

$$y_i\bigr\rvert_{x_i=1} = 5(1 + 1)^2 = 5(2)^2 = 5(4) = 20$$

In [9]:
y = 5 * (x + 1) ** 2
y

tensor([20., 20.], grad_fn=<MulBackward0>)

**Backward should be called only on a scalar (i.e. 1-element tensor) or with gradient w.r.t. the variable**
- Let's reduce y to a scalar then...

$$o = \frac{1}{2}\sum_i y_i$$

In [10]:
o = (1/2) * torch.sum(y)
o

tensor(20., grad_fn=<MulBackward0>)

**Recap `y` equation**: $y_i = 5(x_i+1)^2$

**Recap `o` equation**: $o = \frac{1}{2}\sum_i y_i$

**Substitute `y` into `o` equation**: $o = \frac{1}{2} \sum_i 5(x_i+1)^2$

$$\frac{\partial o}{\partial x_i} = \frac{1}{2}[10(x_i+1)]$$

$$\frac{\partial o}{\partial x_i}\bigr\rvert_{x_i=1} = \frac{1}{2}[10(1 + 1)] = \frac{10}{2}(2) = 10$$

In [11]:
o.backward()

In [12]:
x.grad

tensor([10., 10.])

In [13]:
x.requires_grad

True

In [14]:
y.requires_grad

True

In [15]:
o.requires_grad

True

---
# Summary
- Tensor with Gradients
    - Wraps a tensor for gradient accumulation
- Gradients
    - Define original equation
    - Substitute equation with `x` values
    - Reduce to scalar output, `o` through `mean`
    - Calculate gradients with `o.backward()`
    - Then access gradients of the `x` variable through `x.grad`